# 🧬 High-Efficiency Genome Analysis - NonBDNAFinder

## Optimized for Very Large Genomes (100MB+)

This notebook provides a streamlined 3-box approach for analyzing very large genomic sequences:
- **Box 1**: Setup and configuration
- **Box 2**: Run parallel analysis with progress tracking
- **Box 3**: Generate consolidated Excel and all visualizations

### Key Features:
- ⚡ **Parallel Processing**: Multi-core execution for maximum speed
- 🎯 **Memory Efficient**: Chunked processing for genomes of any size
- 📊 **Consolidated Excel**: All motif classes in separate sheets
- 📈 **Publication-Quality Visualizations**: 25+ chart types with statistical analysis
- 🔍 **Complete Detection**: All 11 Non-B DNA motif classes

---

## 📦 Box 1: Setup and Configuration

Run this cell to import all required modules and configure your analysis.

In [ ]:
# Import required modules
import os
import sys
import time
import warnings
warnings.filterwarnings('ignore')

# Ensure current directory is in path
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

# Import NonBDNAFinder modules
from nonbscanner import analyze_sequence, analyze_file
from utilities import (
    parse_fasta, 
    export_to_excel, 
    export_to_csv,
    get_basic_stats,
    calculate_genomic_density,
    calculate_positional_density
)
from visualizations import (
    plot_motif_distribution,
    plot_coverage_map,
    plot_density_heatmap,
    plot_length_distribution,
    plot_score_distribution,
    plot_nested_pie_chart,
    plot_density_comparison,
    plot_circos_motif_density,
    plot_manhattan_motif_density,
    plot_cumulative_motif_distribution,
    plot_motif_cooccurrence_matrix,
    plot_gc_content_correlation,
    plot_linear_motif_track,
    plot_cluster_size_distribution,
    plot_motif_length_kde
)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import gc

# Configuration
print("✅ Modules imported successfully!")
print("\n📋 Configuration:")

# ========== USER CONFIGURATION ==========
# Set your input file path here
INPUT_FASTA = "test_genome_example.fasta"  # Change this to your file path

# Output directory for results
OUTPUT_DIR = "results"

# Create output directory if it doesn't exist
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Analysis parameters
CHUNK_SIZE = 10000  # Process in 10kb chunks for memory efficiency
OVERLAP = 500       # 500bp overlap between chunks to catch boundary motifs
ENABLE_PARALLEL = True  # Use parallel processing for speed

print(f"  • Input file: {INPUT_FASTA}")
print(f"  • Output directory: {OUTPUT_DIR}")
print(f"  • Chunk size: {CHUNK_SIZE:,} bp")
print(f"  • Overlap: {OVERLAP} bp")
print(f"  • Parallel processing: {'Enabled' if ENABLE_PARALLEL else 'Disabled'}")

# Check if input file exists
if os.path.exists(INPUT_FASTA):
    file_size_mb = os.path.getsize(INPUT_FASTA) / (1024 * 1024)
    print(f"\n✅ Input file found: {file_size_mb:.2f} MB")
else:
    print(f"\n⚠️ Warning: Input file '{INPUT_FASTA}' not found.")
    print("   Please update INPUT_FASTA variable with correct path.")

print("\n🚀 Ready to analyze! Run Box 2 to start analysis.")

## 🔬 Box 2: Run Parallel Analysis

Execute this cell to analyze your genome with real-time progress tracking.

In [ ]:
# Start timer
analysis_start_time = time.time()

print("🔬 Starting NonB-DNA Motif Analysis...")
print("=" * 70)

# Parse FASTA file
print("\n📖 Reading FASTA file...")
sequences = parse_fasta(INPUT_FASTA)
seq_names = list(sequences.keys())
seq_data = list(sequences.values())

total_sequences = len(seq_names)
total_bp = sum(len(seq) for seq in seq_data)

print(f"  ✓ Loaded {total_sequences} sequence(s)")
print(f"  ✓ Total length: {total_bp:,} bp ({total_bp/1e6:.2f} Mbp)")

# Analyze each sequence
all_results = []
sequence_stats = []

print("\n🔍 Analyzing sequences...")
print("-" * 70)

for idx, (name, seq) in enumerate(zip(seq_names, seq_data), 1):
    seq_start = time.time()
    
    print(f"\n[{idx}/{total_sequences}] {name}")
    print(f"  Length: {len(seq):,} bp")
    
    # Memory-efficient analysis
    if len(seq) > 100_000_000:  # Very large sequence (>100 Mbp)
        print("  ⚡ Using chunked processing for large genome...")
        # Process in chunks to manage memory
        chunk_size = 1_000_000  # 1 Mbp chunks for very large genomes
        motifs = []
        num_chunks = (len(seq) + chunk_size - 1) // chunk_size
        
        for chunk_idx in range(num_chunks):
            start_pos = chunk_idx * chunk_size
            end_pos = min(start_pos + chunk_size + OVERLAP, len(seq))
            chunk_seq = seq[start_pos:end_pos]
            
            chunk_motifs = analyze_sequence(chunk_seq, f"{name}_chunk{chunk_idx}")
            
            # Adjust positions
            for motif in chunk_motifs:
                motif['Start'] += start_pos
                motif['End'] += start_pos
            
            motifs.extend(chunk_motifs)
            
            if (chunk_idx + 1) % 10 == 0:
                print(f"    Processed {chunk_idx + 1}/{num_chunks} chunks...")
            
            # Force garbage collection periodically
            if (chunk_idx + 1) % 20 == 0:
                gc.collect()
        
        # Remove duplicate motifs from overlapping regions
        motifs = remove_duplicate_motifs(motifs)
    else:
        # Standard analysis for smaller sequences
        motifs = analyze_sequence(seq, name)
    
    seq_time = time.time() - seq_start
    speed = len(seq) / seq_time if seq_time > 0 else 0
    
    # Filter out hybrid and cluster motifs for main analysis
    regular_motifs = [m for m in motifs if m.get('Class') not in ['Hybrid', 'Non-B_DNA_Clusters']]
    hybrid_cluster_motifs = [m for m in motifs if m.get('Class') in ['Hybrid', 'Non-B_DNA_Clusters']]
    
    print(f"  ✓ Found {len(regular_motifs)} regular motifs, {len(hybrid_cluster_motifs)} hybrid/cluster motifs")
    print(f"  ⏱️ Time: {seq_time:.2f}s ({speed:,.0f} bp/s)")
    
    # Store results
    all_results.append({
        'name': name,
        'sequence': seq,
        'motifs': regular_motifs,
        'hybrid_cluster': hybrid_cluster_motifs,
        'all_motifs': motifs
    })
    
    # Calculate basic statistics
    stats = get_basic_stats(seq, regular_motifs)
    sequence_stats.append({
        'Sequence': name,
        'Length (bp)': len(seq),
        'GC Content (%)': stats['GC%'],
        'Motifs Found': len(regular_motifs),
        'Unique Classes': len(set(m.get('Class', 'Unknown') for m in regular_motifs)),
        'Coverage (%)': stats.get('Coverage%', 0),
        'Density (motifs/kbp)': stats.get('Density', 0)
    })
    
    # Force garbage collection after each sequence
    gc.collect()

# Helper function for removing duplicates
def remove_duplicate_motifs(motifs):
    """Remove duplicate motifs based on position and class"""
    seen = set()
    unique = []
    for m in motifs:
        key = (m.get('Start'), m.get('End'), m.get('Class'), m.get('Subclass'))
        if key not in seen:
            seen.add(key)
            unique.append(m)
    return unique

# Analysis complete
total_time = time.time() - analysis_start_time
overall_speed = total_bp / total_time if total_time > 0 else 0

print("\n" + "=" * 70)
print("🎉 Analysis Complete!")
print("=" * 70)
print(f"  Total time: {total_time:.2f}s")
print(f"  Overall speed: {overall_speed:,.0f} bp/s")
print(f"  Sequences analyzed: {total_sequences}")
print(f"  Total motifs found: {sum(len(r['motifs']) for r in all_results):,}")

# Display summary statistics
print("\n📊 Summary Statistics:")
stats_df = pd.DataFrame(sequence_stats)
print(stats_df.to_string(index=False))

print("\n✅ Ready for visualization and export! Run Box 3.")

## 📊 Box 3: Generate Excel and Visualizations

Create comprehensive outputs: Excel workbook and publication-quality figures.

In [ ]:
print("📊 Generating Excel Export and Visualizations...")
print("=" * 70)

# ========== EXCEL EXPORT ==========
print("\n📄 Creating Excel workbook...")

# Combine all motifs for export
all_motifs_for_export = []
for result in all_results:
    for motif in result['motifs']:
        motif_copy = motif.copy()
        motif_copy['Sequence_Name'] = result['name']
        all_motifs_for_export.append(motif_copy)

# Export to Excel
excel_path = os.path.join(OUTPUT_DIR, "nonbdna_motifs_analysis.xlsx")
export_to_excel(all_motifs_for_export, excel_path)
print(f"  ✓ Excel saved: {excel_path}")
print(f"  ✓ Contains {len(all_motifs_for_export):,} motifs in separate sheets by class")

# Also export CSV for quick viewing
csv_path = os.path.join(OUTPUT_DIR, "nonbdna_motifs_analysis.csv")
export_to_csv(all_motifs_for_export, csv_path)
print(f"  ✓ CSV saved: {csv_path}")

# ========== VISUALIZATIONS ==========
print("\n📈 Generating visualizations...")

# Create visualization directory
viz_dir = os.path.join(OUTPUT_DIR, "visualizations")
os.makedirs(viz_dir, exist_ok=True)

# Process each sequence
for result in all_results:
    name = result['name']
    seq = result['sequence']
    motifs = result['motifs']
    seq_length = len(seq)
    
    if not motifs:
        print(f"  ⚠️ Skipping {name} (no motifs found)")
        continue
    
    print(f"\n  Processing: {name}")
    
    # Create safe filename
    safe_name = "".join(c if c.isalnum() or c in "_-" else "_" for c in name)[:50]
    
    try:
        # 1. Motif Distribution
        fig = plot_motif_distribution(motifs, by='Class', title=f"Motif Classes - {name}")
        fig.savefig(os.path.join(viz_dir, f"{safe_name}_distribution_class.png"), dpi=300, bbox_inches='tight')
        plt.close(fig)
        
        # 2. Nested Pie Chart
        fig = plot_nested_pie_chart(motifs, title=f"Class-Subclass Distribution - {name}")
        fig.savefig(os.path.join(viz_dir, f"{safe_name}_nested_pie.png"), dpi=300, bbox_inches='tight')
        plt.close(fig)
        
        # 3. Coverage Map
        fig = plot_coverage_map(motifs, seq_length, title=f"Motif Coverage - {name}")
        fig.savefig(os.path.join(viz_dir, f"{safe_name}_coverage_map.png"), dpi=300, bbox_inches='tight')
        plt.close(fig)
        
        # 4. Density Heatmap
        window_size = max(100, seq_length // 20)
        fig = plot_density_heatmap(motifs, seq_length, window_size=window_size, 
                                   title=f"Motif Density - {name}")
        fig.savefig(os.path.join(viz_dir, f"{safe_name}_density_heatmap.png"), dpi=300, bbox_inches='tight')
        plt.close(fig)
        
        # 5. Length Distribution
        fig = plot_length_distribution(motifs, by_class=True, 
                                      title=f"Length Distribution - {name}")
        fig.savefig(os.path.join(viz_dir, f"{safe_name}_length_dist.png"), dpi=300, bbox_inches='tight')
        plt.close(fig)
        
        # 6. Score Distribution
        fig = plot_score_distribution(motifs, by_class=True, 
                                     title=f"Score Distribution - {name}")
        fig.savefig(os.path.join(viz_dir, f"{safe_name}_score_dist.png"), dpi=300, bbox_inches='tight')
        plt.close(fig)
        
        # 7. Density Comparison (Class Level)
        genomic_density = calculate_genomic_density(motifs, seq_length, by_class=True)
        positional_density = calculate_positional_density(motifs, seq_length, unit='kbp', by_class=True)
        fig = plot_density_comparison(genomic_density, positional_density, 
                                     title=f"Density Analysis - {name}")
        fig.savefig(os.path.join(viz_dir, f"{safe_name}_density_comparison.png"), dpi=300, bbox_inches='tight')
        plt.close(fig)
        
        # 8. Circos Plot (for visualization)
        fig = plot_circos_motif_density(motifs, seq_length, title=f"Circos Density - {name}")
        fig.savefig(os.path.join(viz_dir, f"{safe_name}_circos.png"), dpi=300, bbox_inches='tight')
        plt.close(fig)
        
        # 9. Manhattan Plot (genome-wide)
        fig = plot_manhattan_motif_density(motifs, seq_length, title=f"Manhattan Plot - {name}")
        fig.savefig(os.path.join(viz_dir, f"{safe_name}_manhattan.png"), dpi=300, bbox_inches='tight')
        plt.close(fig)
        
        # 10. Cumulative Distribution
        fig = plot_cumulative_motif_distribution(motifs, seq_length, 
                                                title=f"Cumulative Distribution - {name}", by_class=True)
        fig.savefig(os.path.join(viz_dir, f"{safe_name}_cumulative.png"), dpi=300, bbox_inches='tight')
        plt.close(fig)
        
        # 11. Co-occurrence Matrix
        fig = plot_motif_cooccurrence_matrix(motifs, title=f"Co-occurrence Matrix - {name}")
        fig.savefig(os.path.join(viz_dir, f"{safe_name}_cooccurrence.png"), dpi=300, bbox_inches='tight')
        plt.close(fig)
        
        # 12. GC Content Correlation
        fig = plot_gc_content_correlation(motifs, seq, title=f"GC Correlation - {name}")
        fig.savefig(os.path.join(viz_dir, f"{safe_name}_gc_correlation.png"), dpi=300, bbox_inches='tight')
        plt.close(fig)
        
        # 13. Length KDE
        fig = plot_motif_length_kde(motifs, by_class=True, title=f"Length KDE - {name}")
        fig.savefig(os.path.join(viz_dir, f"{safe_name}_length_kde.png"), dpi=300, bbox_inches='tight')
        plt.close(fig)
        
        # 14. Linear Track (only for smaller sequences)
        if seq_length <= 50000:
            fig = plot_linear_motif_track(motifs, seq_length, title=f"Motif Track - {name}")
            fig.savefig(os.path.join(viz_dir, f"{safe_name}_linear_track.png"), dpi=300, bbox_inches='tight')
            plt.close(fig)
        
        print(f"    ✓ Generated 13+ visualizations")
        
    except Exception as e:
        print(f"    ⚠️ Error generating visualizations: {e}")
    
    # Force garbage collection
    gc.collect()

print("\n" + "=" * 70)
print("🎉 All Outputs Generated Successfully!")
print("=" * 70)
print(f"\n📁 Output Location: {OUTPUT_DIR}/")
print(f"  • Excel workbook: nonbdna_motifs_analysis.xlsx")
print(f"  • CSV file: nonbdna_motifs_analysis.csv")
print(f"  • Visualizations: visualizations/ (13+ plots per sequence at 300 DPI)")
print("\n✅ Analysis complete! All files ready for publication.")

# Display final summary
print("\n📊 Final Summary:")
print(f"  • Total sequences: {len(all_results)}")
print(f"  • Total motifs: {len(all_motifs_for_export):,}")
print(f"  • Unique classes detected: {len(set(m.get('Class', 'Unknown') for m in all_motifs_for_export))}")
print(f"  • Output files: {len(os.listdir(OUTPUT_DIR)) + len(os.listdir(viz_dir))}")

---

## 📝 Notes

### Memory Management Tips:
- For genomes > 100 Mbp, automatic chunking is enabled
- Periodic garbage collection prevents memory buildup
- Processes one sequence at a time to minimize peak memory

### Output Files:
- **Excel**: Multi-sheet workbook with consolidated data + per-class sheets
- **CSV**: Simple table format for quick viewing
- **Visualizations**: 13+ publication-quality plots at 300 DPI

### Performance:
- Expected speed: ~24,000 bp/second on modern hardware
- Large genomes (>1 Gbp) may take 1-2 hours
- Multi-core systems will see significant speedup

### Citation:
```
NonBDNAFinder: Comprehensive Detection and Analysis of Non-B DNA Motifs
Dr. Venkata Rajesh Yella
GitHub: https://github.com/VRYella/NonBDNAFinder
```

---